In [ ]:
# Heavy-Weight LLM in Google Colab with GPU/TPU

This notebook demonstrates how to run a large language model (LLM) in Google Colab with GPU and TPU acceleration.

**Setup Instructions:**
1. Save this notebook to Google Colab
2. Go to **Runtime** → **Change runtime type** → Select **GPU** or **TPU**
3. Click **Save**
4. Run cells in order

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!nvidia-smi

In [ ]:
# ============================================
# Step 1: Check GPU/TPU Availability
# ============================================
# !pip install torch tensorflow
import torch
import tensorflow as tf

print("=" * 60)
print("HARDWARE ACCELERATION CHECK")
print("=" * 60)

# Check CUDA/GPU availability
print("\n🔹 GPU Information:")
print(f"   CUDA Available: {torch.cuda.is_available()}")
print(f"   Device Count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"   Device Name: {torch.cuda.get_device_name(0)}")
    print(f"   Device Capability: {torch.cuda.get_device_capability(0)}")
    
# Check TensorFlow GPU
print("\n🔹 TensorFlow GPU:")
gpus = tf.config.list_physical_devices('GPU')
print(f"   Physical GPUs: {len(gpus)}")
if gpus:
    for gpu in gpus:
        print(f"   - {gpu}")

# Check TPU availability
print("\n🔹 TPU Information:")
try:
    import tensorflow as tf
    tpu_resolver = tf.distribute.cluster_resolver.TPUClusterResolver()
    print(f"   TPU Available: True")
    print(f"   TPU Master: {tpu_resolver.master()}")
    tpu_strategy = tf.distribute.TPUStrategy(tpu_resolver)
    print(f"   TPU Strategy initialized: Yes")
except Exception as e:
    print(f"   TPU Available: False (This is normal for GPU runtime)")
    print(f"   Details: {str(e)[:100]}...")

print("\n✅ Hardware check complete!")
print("=" * 60)

In [ ]:
# ============================================
# Step 2: Install Required Packages
# ============================================

import subprocess
import sys

print("📦 Installing required packages...")
print("=" * 60)

packages = [
    "torch>=2.0.0",           # PyTorch with CUDA support (auto-installed in Colab)
    "transformers>=4.35.0",   # Hugging Face transformers
    "accelerate>=0.24.0",     # Model optimization
    "bitsandbytes>=0.41.0",   # 8-bit quantization for memory efficiency
    "peft>=0.7.0",            # Parameter-efficient fine-tuning
]

for package in packages:
    print(f"\n   Installing: {package}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("\n✅ All packages installed!")
print("=" * 60)

In [ ]:
# ============================================
# Step 3: Load Heavy-Weight LLM with Optimization
# ============================================

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

print("🤖 Loading LLM Model with Optimizations...")
print("=" * 60)

# Model selection (using efficient quantized versions)
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"  # Fast and efficient 7B model
# Alternative heavy models:
# - "meta-llama/Llama-2-13b-chat-hf"  # 13B (requires HF token)
# - "meta-llama/Llama-2-70b-chat-hf"  # 70B (needs multiple GPUs)
# - "TheBloke/Mistral-7B-Instruct-v0.1-GPTQ"  # Quantized version

print(f"\n📥 Model: {MODEL_ID}")

# ✅ Quantization config (reduces memory by 75%)
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,                    # 8-bit quantization
    bnb_4bit_use_double_quant=True,      # Double quant for extra savings
    bnb_4bit_quant_type="nf4",           # Normal float 4
    bnb_4bit_compute_dtype=torch.float16, # Compute in FP16
)

print("   Quantization: 8-bit + NF4 (saves ~75% memory)")

try:
    # Load tokenizer
    print("\n   Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    tokenizer.pad_token = tokenizer.eos_token
    
    # Load model with quantization
    print("   Loading model with 8-bit quantization...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",  # Automatically place on GPU/TPU
        trust_remote_code=True,
    )
    
    # Get model info
    total_params = sum(p.numel() for p in model.parameters())
    print(f"\n✅ Model loaded successfully!")
    print(f"   Total parameters: {total_params / 1e9:.2f}B")
    print(f"   Model dtype: {model.dtype}")
    print(f"   Device: {next(model.parameters()).device}")
    
except Exception as e:
    print(f"\n❌ Error loading model: {e}")
    print("\n   Fallback: Loading smaller model...")
    
    # Fallback to smaller model if loading fails
    MODEL_ID = "gpt2-medium"
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID)
    print(f"   ✅ Loaded fallback model: {MODEL_ID}")

print("=" * 60)

In [ ]:
# ============================================
# Step 4: Run Inference on LLM
# ============================================

import torch
import time

print("🚀 Running Inference...")
print("=" * 60)

# Test prompts
prompts = [
    "What is a good strategy for machine learning?",
    "Explain quantum computing in simple terms.",
    "Write a Python function to count words in a string.",
]

def generate_text(prompt, max_length=100):
    """Generate text using the loaded LLM"""
    print(f"\n📝 Input: {prompt}")
    print("-" * 60)
    
    # Tokenize input
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # Record time
    start_time = time.time()
    
    with torch.no_grad():
        # Generate output
        outputs = model.generate(
            inputs.input_ids,
            max_length=max_length,
            temperature=0.7,           # Control randomness
            top_p=0.9,                # Nucleus sampling
            do_sample=True,           # Use sampling instead of greedy
            pad_token_id=tokenizer.eos_token_id,
        )
    
    elapsed = time.time() - start_time
    
    # Decode output
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"🤖 Output:\n{generated_text}")
    print(f"\n⏱️  Generation time: {elapsed:.2f}s")
    
    return generated_text

# Run inference on first prompt
print("\n🔄 Running first inference (may take longer)...")
result = generate_text(prompts[0], max_length=80)

print("\n✅ Inference complete!")
print("=" * 60)

In [ ]:
# ============================================
# Step 5: Batch Inference (Process Multiple Inputs)
# ============================================

print("📊 Running Batch Inference...")
print("=" * 60)

prompts_batch = [
    "What is machine learning?",
    "How does deep learning work?",
    "Explain neural networks.",
]

def batch_generate(prompts, max_length=100):
    """Generate text for multiple prompts efficiently"""
    results = []
    
    for i, prompt in enumerate(prompts, 1):
        print(f"\n[{i}/{len(prompts)}] Processing: {prompt[:40]}...")
        
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                inputs.input_ids,
                max_length=max_length,
                temperature=0.7,
                top_p=0.9,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
            )
        
        generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
        results.append(generated)
        print(f"   ✅ Generated {len(generated)} characters")
    
    return results

# Run batch inference
batch_results = batch_generate(prompts_batch, max_length=80)

print(f"\n✅ Processed {len(batch_results)} prompts successfully!")
print("=" * 60)

In [ ]:
# ============================================
# Step 6: Memory Management & GPU Utilization
# ============================================

import psutil
import os

print("💾 Memory & GPU Utilization Stats")
print("=" * 60)

# CPU Memory
process = psutil.Process(os.getpid())
cpu_memory = process.memory_info().rss / 1024 / 1024 / 1024  # Convert to GB

print(f"\n🖥️  CPU Memory Usage:")
print(f"   Notebook (Python): {cpu_memory:.2f} GB")

# GPU Memory
if torch.cuda.is_available():
    print(f"\n📈 GPU Memory Usage:")
    print(f"   GPU 0:")
    print(f"   - Allocated: {torch.cuda.memory_allocated(0) / 1024 / 1024 / 1024:.2f} GB")
    print(f"   - Reserved: {torch.cuda.memory_reserved(0) / 1024 / 1024 / 1024:.2f} GB")
    print(f"   - Total: {torch.cuda.get_device_properties(0).total_memory / 1024 / 1024 / 1024:.2f} GB")

# Memory optimization tips
print("\n" + "=" * 60)
print("💡 Memory Optimization Tips:")
print("=" * 60)
tips = """
1. 🔄 Use 8-bit or 4-bit quantization (already enabled)
   - Reduces memory by 50-75%
   - Minimal quality loss

2. ⚡ Enable Flash Attention (for Transformers 4.35+)
   - Speeds up inference by 2-3x
   - Use: model = AutoModelForCausalLM.from_pretrained(..., 
                                            attn_implementation="flash_attention_2")

3. 🎯 Use LoRA for fine-tuning (Parameter Efficient)
   - Train with <1% of parameters
   - Saves memory and time

4. 🧠 Batch inference instead of sequential
   - Process multiple inputs at once
   - More efficient GPU utilization

5. 🗑️  Clear cache between generations
   - torch.cuda.empty_cache()
   - Prevents memory fragmentation

6. ⚙️  Reduce max_length parameter
   - Faster generation
   - Less memory usage

7. 📊 Use device_map="auto"
   - Automatically splits model across GPU/CPU if needed
   - Handles very large models

8. 🔧 Compile model (PyTorch 2.0+)
   - model = torch.compile(model)
   - 10-20% faster inference
"""
print(tips)
print("=" * 60)

In [ ]:
# ============================================
# Step 7: Advanced - LoRA Fine-tuning (Optional)
# ============================================

from peft import get_peft_model, LoraConfig, TaskType

print("🎓 Parameter-Efficient Fine-Tuning with LoRA")
print("=" * 60)

print("\n📝 LoRA Configuration:")
print("   LoRA (Low-Rank Adaptation) allows fine-tuning with <1% of parameters")

# Configure LoRA
lora_config = LoraConfig(
    r=16,                              # LoRA rank
    lora_alpha=32,                    # LoRA alpha
    target_modules=["q_proj", "v_proj"],  # Modules to apply LoRA
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

print("\n🔧 Applying LoRA to model...")
try:
    # Apply LoRA adapter
    model_lora = get_peft_model(model, lora_config)
    
    trainable_params = sum(p.numel() for p in model_lora.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model_lora.parameters())
    
    print(f"✅ LoRA Applied Successfully!")
    print(f"   Trainable params: {trainable_params / 1e6:.2f}M")
    print(f"   Total params: {total_params / 1e6:.2f}M")
    print(f"   Trainable %: {100 * trainable_params / total_params:.2f}%")
    
    model_lora.print_trainable_parameters()
    
except Exception as e:
    print(f"⚠️  LoRA setup error: {e}")
    print("   (This is normal if you want to just use the base model)")

print("\n" + "=" * 60)
print("✨ Setup Complete! You can now:")
print("   1. Run inference with the model")
print("   2. Fine-tune with LoRA (if applied)")
print("   3. Use GPU/TPU acceleration")
print("   4. Process large amounts of text efficiently")
print("=" * 60)

## Quick Reference & Troubleshooting

### 🎯 Key Features Implemented

| Feature | Benefit | Status |
|---------|---------|--------|
| **8-bit Quantization** | 75% memory reduction | ✅ Enabled |
| **GPU/TPU Auto-detection** | Automatic hardware utilization | ✅ Enabled |
| **Batch Processing** | Efficient multi-prompt inference | ✅ Enabled |
| **LoRA Fine-tuning** | Train with <1% parameters | ✅ Optional |
| **Memory Monitoring** | Real-time memory stats | ✅ Enabled |

### ⚙️ Common Issues & Solutions

**Issue: CUDA Out of Memory (OOM)**
- ✅ Solution: Already using 8-bit quantization
- ✅ Alternative: Use smaller model or reduce `max_length`
- ✅ Reset: Run `torch.cuda.empty_cache()`

**Issue: Model loading is slow**
- ✅ This is normal for first run (downloads ~14GB for Mistral 7B)
- ✅ Subsequent runs use cache (instant loading)

**Issue: TPU not available in GPU runtime**
- ✅ This is expected (TPU and GPU are different runtimes)
- ✅ GPU is faster for most LLM inference tasks

**Issue: Timeout during generation**
- ✅ Reduce `max_length` parameter
- ✅ Increase timeout: Add `timeout=60` to generate call

### 🚀 Model Recommendations

| Model | Size | Speed | Quality | Use Case |
|-------|------|-------|---------|----------|
| Mistral 7B | 7B | ⚡⚡⚡ | ⭐⭐⭐⭐ | **Default (Best)** |
| Llama 2 13B | 13B | ⚡⚡ | ⭐⭐⭐⭐⭐ | High quality, requires token |
| GPT2 Medium | 350M | ⚡⚡⚡⚡ | ⭐⭐ | Testing, very fast |
| Llama 2 70B | 70B | ⚡ | ⭐⭐⭐⭐⭐ | Only with multiple GPUs |

### 💾 Memory Requirements

- **Mistral 7B (8-bit)**: ~7-9 GB VRAM
- **Llama 13B (8-bit)**: ~14-16 GB VRAM  
- **Llama 70B (8-bit)**: ~64+ GB (needs multiple GPUs or TPU)

**Colab Free GPU**: ~16 GB (sufficient for 7-13B models)

### 📚 Useful Resources

- [Hugging Face Model Hub](https://huggingface.co/models)
- [Transformers Documentation](https://huggingface.co/docs/transformers/)
- [PEFT (LoRA) Guide](https://github.com/huggingface/peft)
- [bitsandbytes Quantization](https://github.com/TimDettmers/bitsandbytes)

### ✨ Next Steps

1. **Click** "Runtime" → "Change Runtime Type" → Select **GPU**
2. **Run** all cells from top to bottom
3. **Customize** prompts and parameters as needed
4. **Share** results and findings

---

**Happy LLMing! 🚀**